## **P3: Exploratory Data Analysis**

Performing exploratory data analysis on the processed leukemia gene expression dataset, includes:
- Loading and combining cleaned features and labels
- Computing descriptive statistics
- Grouping analysis by leukemia type
- ANOVA testing for significant gene expression differences
- Correlation analysis between genes

### **1. Import Required Libraries**

Load all necessary libraries for data analysis, visualization, and statistical testing.

In [121]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 10)

### **2. Load and Combine Data**

Load the cleaned feature matrix and labels from the data wrangling step. Combine them into a single dataframe with samples as rows and gene expression values as columns, aligned by Sample_ID.

In [122]:
# load the 2 cleaned csv files
features_df = pd.read_csv('GSE13164_cleaned_features.csv', index_col=0)
labels_df = pd.read_csv('GSE13164_cleaned_labels.csv')

# align & combine: Sample_ID | Leukemia_Type | Gene1 ... geneN
combined_df = (
    labels_df[['Sample_ID', 'Leukemia_Type']]
    .set_index('Sample_ID')
    .join(features_df, how='inner')
    .reset_index()
)

# reorder cols: Sample_ID & Leukemia_Type, then genes
combined_df = combined_df[['Sample_ID', 'Leukemia_Type'] + [c for c in combined_df.columns if c not in ['Sample_ID', 'Leukemia_Type']]]

print("Data successfully combined!")
print(f"Combined dataset shape: {combined_df.shape}")

Data successfully combined!
Combined dataset shape: (973, 1422)


### **3. Basic Data Overview**

Display basic information about the dataset including shape, dimensions, and sample rows to verify data integrity and structure.

In [123]:
print(f"Dataset Shape: {combined_df.shape}")
print(f"Columns: Sample_ID, Leukemia_Type, {combined_df.shape[1]-2} genes")


Dataset Shape: (973, 1422)
Columns: Sample_ID, Leukemia_Type, 1420 genes


In [124]:
print("First 5 rows (first 10 columns):")
combined_df.head(5)

First 5 rows (first 10 columns):


,Sample_ID,Leukemia_Type,AA022668,AA045174,AA045184,...,X95660,Y00062,Z22969,Z25521,Z49258
0,GSM331733,ALL,0.021524,0.313375,0.621374,...,0.147316,0.652401,0.251543,0.542704,0.034048
1,GSM331735,ALL,0.138425,0.367138,0.209544,...,0.161024,0.643809,0.166796,0.470802,0.048288
2,GSM331736,ALL,0.166461,0.353492,0.584845,...,0.100688,0.548033,0.100538,0.508992,0.076684
3,GSM331737,ALL,0.119696,0.390152,0.206351,...,0.079367,0.780182,0.147899,0.711219,0.036944
4,GSM331739,ALL,0.072855,0.344026,0.110417,...,0.125102,0.561885,0.119055,0.479710,0.075409


In [125]:
print("Last 5 rows (excluding last gene column):")
combined_df.tail(5)

Last 5 rows (excluding last gene column):


,Sample_ID,Leukemia_Type,AA022668,AA045174,AA045184,...,X95660,Y00062,Z22969,Z25521,Z49258
968,GSM332879,CLL,0.126125,0.457595,0.182019,...,0.140510,0.741261,0.345432,0.590160,0.139727
969,GSM332880,AML,0.008868,0.451833,0.218862,...,0.115515,0.719811,0.017433,0.623884,0.233855
970,GSM332881,ALL,0.046805,0.324388,0.251160,...,0.090959,0.509765,0.197561,0.411354,0.058076
971,GSM332882,AML,0.063562,0.397079,0.223211,...,0.157383,0.606927,0.047135,0.542704,0.064848
972,GSM332883,CLL,0.038635,0.585898,0.226493,...,0.200107,0.738707,0.146915,0.535772,0.115851


In [126]:
# separate features from metadata for later use
gene_expression = combined_df.iloc[:, 2:]  # All columns after Sample_ID and Leukemia_Type
leukemia_types = combined_df['Leukemia_Type']

print(f"Gene expression matrix shape: {gene_expression.shape}")
print(f"Leukemia types: {sorted(leukemia_types.unique())}")

Gene expression matrix shape: (973, 1420)
Leukemia types: ['ALL', 'AML', 'CLL', 'CML']


### **4. Descriptive Statistics**

Compute and display summary statistics (mean, std, min, max...) for gene expression data both overall and stratified by leukemia type.

In [127]:
print("Overall gene expression statistics:")
gene_expression.describe()

Overall gene expression statistics:


,AA022668,AA045174,AA045184,AA054642,AA058578,...,X95660,Y00062,Z22969,Z25521,Z49258
count,973.000000,973.000000,973.000000,973.000000,973.000000,...,973.000000,973.000000,973.000000,973.000000,973.000000
mean,0.075085,0.380579,0.329312,0.184106,0.371812,...,0.139644,0.665124,0.204221,0.550737,0.112663
std,0.076518,0.092361,0.186626,0.114658,0.081254,...,0.036342,0.095544,0.148617,0.054515,0.079998
min,0.000000,0.029028,0.024423,0.000000,0.044819,...,0.011156,0.324388,0.000000,0.326625,0.000000
25%,0.024423,0.321276,0.201414,0.089093,0.316431,...,0.115329,0.612824,0.087186,0.516030,0.058723
50%,0.053365,0.370126,0.257699,0.166126,0.356376,...,0.137917,0.691644,0.165457,0.548033,0.097993
75%,0.098629,0.440007,0.432756,0.266284,0.419293,...,0.163679,0.733738,0.304334,0.581717,0.148228
max,0.558113,0.615230,0.770034,0.609266,0.608093,...,0.274818,0.828838,0.776699,0.749232,0.757725


In [128]:
type = 'ALL'
print(f"ALL (n={(leukemia_types == type).sum()}):")
gene_expression[leukemia_types == type].describe().loc[['mean', 'std', 'min', 'max']]

ALL (n=436):


,AA022668,AA045174,AA045184,AA054642,AA058578,...,X95660,Y00062,Z22969,Z25521,Z49258
mean,0.095957,0.363487,0.436750,0.138011,0.350505,...,0.133457,0.605043,0.129496,0.545555,0.094267
std,0.097490,0.066484,0.212278,0.088745,0.054624,...,0.035382,0.103888,0.104247,0.065438,0.057564
min,0.000000,0.094814,0.024423,0.000000,0.110098,...,0.036692,0.324388,0.000000,0.375666,0.005058
max,0.558113,0.584845,0.770034,0.520022,0.539214,...,0.234887,0.817999,0.602326,0.736200,0.370626


In [129]:
type = 'AML'
print(f"ALL (n={(leukemia_types == type).sum()}):")
gene_expression[leukemia_types == type].describe().loc[['mean', 'std', 'min', 'max']]

ALL (n=257):


,AA022668,AA045174,AA045184,AA054642,AA058578,...,X95660,Y00062,Z22969,Z25521,Z49258
mean,0.058559,0.324695,0.267990,0.188743,0.323175,...,0.137078,0.693305,0.264867,0.556046,0.130914
std,0.048464,0.079834,0.120176,0.131798,0.059026,...,0.033782,0.054065,0.170764,0.054932,0.110309
min,0.000000,0.029028,0.040988,0.009505,0.044819,...,0.011156,0.471477,0.002547,0.326625,0.000000
max,0.295888,0.535772,0.728942,0.609266,0.524893,...,0.250126,0.817999,0.776699,0.749232,0.757725


In [130]:
type = 'CLL'
print(f"ALL (n={(leukemia_types == type).sum()}):")
gene_expression[leukemia_types == type].describe().loc[['mean', 'std', 'min', 'max']]

ALL (n=237):


,AA022668,AA045174,AA045184,AA054642,AA058578,...,X95660,Y00062,Z22969,Z25521,Z49258
mean,0.053838,0.484737,0.216842,0.245254,0.472142,...,0.154386,0.742073,0.268646,0.552971,0.128241
std,0.045128,0.063525,0.058821,0.098931,0.061662,...,0.037323,0.028854,0.137048,0.028183,0.067791
min,0.000000,0.299248,0.039981,0.003558,0.303908,...,0.071494,0.674193,0.000000,0.464787,0.003599
max,0.221395,0.615230,0.423914,0.448051,0.608093,...,0.274818,0.828838,0.695368,0.645214,0.339838


In [131]:
type = 'CML'
print(f"ALL (n={(leukemia_types == type).sum()}):")
gene_expression[leukemia_types == type].describe().loc[['mean', 'std', 'min', 'max']]

ALL (n=43):


,AA022668,AA045174,AA045184,AA054642,AA058578,...,X95660,Y00062,Z22969,Z25521,Z49258
mean,0.079326,0.313815,0.226332,0.286756,0.325575,...,0.136456,0.681760,0.244340,0.559224,0.104259
std,0.050441,0.046864,0.082445,0.087899,0.033355,...,0.033797,0.031008,0.098916,0.031924,0.082489
min,0.007460,0.186818,0.053202,0.110098,0.246972,...,0.049934,0.617665,0.010546,0.505159,0.013965
max,0.206351,0.419293,0.562838,0.474196,0.393333,...,0.193398,0.752001,0.435155,0.620130,0.354451


### **5. Grouping Analysis**

Analyze gene expression patterns grouped by leukemia type. Compute mean expression for each gene within each leukemia class and show sample distributions.

In [132]:
grouped_stats = combined_df.groupby('Leukemia_Type')[gene_expression.columns].mean()
print("Mean gene expression by leukemia type:")
grouped_stats

Mean gene expression by leukemia type:


,AA022668,AA045174,AA045184,AA054642,AA058578,...,X95660,Y00062,Z22969,Z25521,Z49258
Leukemia_Type,,,,,,,,,,,
ALL,0.095957,0.363487,0.436750,0.138011,0.350505,...,0.133457,0.605043,0.129496,0.545555,0.094267
AML,0.058559,0.324695,0.267990,0.188743,0.323175,...,0.137078,0.693305,0.264867,0.556046,0.130914
CLL,0.053838,0.484737,0.216842,0.245254,0.472142,...,0.154386,0.742073,0.268646,0.552971,0.128241
CML,0.079326,0.313815,0.226332,0.286756,0.325575,...,0.136456,0.681760,0.244340,0.559224,0.104259


In [133]:
sample_counts = combined_df.groupby('Leukemia_Type').size()
print("Sample counts by leukemia type:")
sample_counts

Sample counts by leukemia type:


Leukemia_Type
ALL    436
AML    257
CLL    237
CML     43
dtype: int64

### **6. ANOVA Testing**

Perform one-way ANOVA test to determine if gene expression differs significantly across leukemia types. Compute F-statistics and p-values for each gene to identify genes with significant differential expression.

In [134]:
f_stats = []
p_values = []

for gene in gene_expression.columns:
    groups = [gene_expression.loc[leukemia_types == ltype, gene].values 
              for ltype in sorted(leukemia_types.unique())]
    f_stat, p_val = stats.f_oneway(*groups)
    f_stats.append(f_stat)
    p_values.append(p_val)

# create results df
anova_df = pd.DataFrame({
    'Gene': gene_expression.columns,
    'F-Statistic': f_stats,
    'P-Value': p_values
}).sort_values('P-Value')

In [135]:
# summary stats
print(f"Mean F-statistic across genes: {np.nanmean(f_stats):.4f}")
print(f"Mean P-value across genes: {np.nanmean(p_values):.4e}")
print(f"Minimum P-value across genes: {np.nanmin(p_values):.4e}")

Mean F-statistic across genes: 170.8113
Mean P-value across genes: 1.0557e-02
Minimum P-value across genes: 0.0000e+00


In [136]:
if np.nanmin(p_values) < 0.05:
    print("At least one gene shows significant differences across leukemia types (min p < 0.05)")
else:
    print("No genes show significant differences across leukemia types (min p >= 0.05)")

At least one gene shows significant differences across leukemia types (min p < 0.05)


In [137]:
print("Top 5 genes with most and least significant differences by leukemia type:")
anova_df

Top 5 genes with most and least significant differences by leukemia type:


,Gene,F-Statistic,P-Value
748,BG528420,1373.353380,0.000000
370,AK025578,1579.511658,0.000000
1191,NM_014736,1587.628286,0.000000
803,M11722,1252.224851,0.000000
911,NM_001071,1845.146364,0.000000
...,...,...,...
448,AL568652,0.455956,0.713145
557,AW950865,0.375386,0.770781
251,AI539425,0.332968,0.801517
129,AF071542,0.268357,0.848234


### **7. Correlation Analysis**

Compute the correlation matrix between all genes to identify relationships and potential co-expression patterns. Display gene pairs with highest and lowest correlations to understand inter-gene dependencies.

In [138]:
correlation_matrix = gene_expression.corr()
print(f"Correlation matrix shape: {correlation_matrix.shape}")

Correlation matrix shape: (1420, 1420)


In [139]:
# get gene pairs and their correlations
corr_pairs = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        corr_pairs.append({
            'Gene1': correlation_matrix.columns[i],
            'Gene2': correlation_matrix.columns[j],
            'Correlation': correlation_matrix.iloc[i, j]
        })

corr_pairs_df = pd.DataFrame(corr_pairs).sort_values('Correlation', ascending=False)

print("Top 5 gene pairs with highest and lowest correlation:")
corr_pairs_df

Top 5 gene pairs with highest and lowest correlation:


,Gene1,Gene2,Correlation
76851,AA761181,AK000168,0.995127
909093,NM_002125,U65585,0.992458
996854,NM_021983,U65585,0.991755
851621,N90866,NM_001803,0.991656
670290,BC005332,M63438,0.991542
...,...,...,...
92633,AA838075,NM_014736,-0.877441
81501,AA769410,NM_001071,-0.886569
878472,NM_001071,NM_014383,-0.887038
81686,AA769410,NM_005375,-0.887646


### **8. Analysis Summary**

Display summary statistics and key findings from the exploratory data analysis.

In [140]:
print(f"Samples: {len(combined_df)}")
print(f"Genes: {len(gene_expression.columns)}")
print(f"Leukemia types: {len(leukemia_types.unique())}")
print(f"Significant genes (p < 0.05): {(anova_df['P-Value'] < 0.05).sum()}")
print(f"Total gene pairs: {len(corr_pairs_df)}")

Samples: 973
Genes: 1420
Leukemia types: 4
Significant genes (p < 0.05): 1378
Total gene pairs: 1007490
